# Notebook 03: Validación y Contratos de Datos (Data Quality Gates)
**Proyecto**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Fase PDCO**: DEVELOPMENT $\rightarrow$ CONTROL | **Capa**: Bronze $\rightarrow$ Silver Quality Gate  
**Estándar**: DAMA-DMBOK 2 / ISO/IEC 25010 / SWEBOK v4  

---
### Objetivo del Quality Gate
Aplicar los contratos formales de validación sobre las extracciones crudas de `data/RAW/`:
1. Validar rangos físicos y restricciones lógicas (lluvia $\ge 0$, temperaturas en límites páramo/desierto, área cosechada $\le$ área sembrada, precios $> 0$).
2. Particionar los registros en subconjuntos conformes y no conformes.
3. Auditar la unicidad de llaves compuestas primarias antes de la normalización hacia la capa Silver.

In [1]:
# Celda 1: Setup e Importaciones de Contratos
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.settings import settings
from src.utils.logger import get_logger
from src.ingestion.landing_manager import LandingManager
from src.validation.contracts import (
    EVARecordContract,
    IDEAMClimaContract,
    SIPSAPrecioContract,
    ICAPecuarioContract,
)
from src.validation.quality_checker import QualityChecker

logger = get_logger("notebooks.03_validation")
landing = LandingManager()
manifest = landing.load_manifest()
print(f"Cargando datasets desde Bronze. Registrados: {len(manifest.get('datasets', {}))}")

Cargando datasets desde Bronze. Registrados: 11


In [2]:
# Celda 2: Quality Gate para Evaluaciones Agropecuarias Municipales (UPRA EVA)
eva_path = settings.RAW_DIR / "upra" / "upra_eva_historico.parquet"
if eva_path.exists():
    df_eva = pd.read_parquet(eva_path)
    valid_eva, invalid_eva, metrics_eva = QualityChecker.validate_dataset(df_eva, EVARecordContract)
    print(QualityChecker.generate_markdown_report(metrics_eva, "UPRA EVA Histórico"))
else:
    print("Dataset UPRA EVA no encontrado en Bronze. Ejecuta previamente el Notebook 01.")

2026-10-07T19:39:10-0500 | INFO     | agrostats.validation.quality_checker:validate_dataset:76 | Validación EVARecordContract completada: 5000/5000 conformes (100.00%)


### Reporte de Quality Gate: UPRA EVA Histórico
- **Estado de Puerta de Calidad**: `PASÓ (GREEN)`
- **Total Registros Evaluados**: `5000`
- **Registros Conformes**: `5000`
- **Registros No Conformes**: `0`
- **Tasa de Conformidad**: `100.0%`

#### Desglose de No Conformidades:
*Cero violaciones detectadas.*


In [3]:
# Celda 3: Quality Gate para Mediciones Meteorológicas (IDEAM)
ideam_files = list((settings.RAW_DIR / "ideam").glob("*.parquet"))
for fpath in ideam_files:
    df_ideam = pd.read_parquet(fpath)
    valid_cli, invalid_cli, metrics_cli = QualityChecker.validate_dataset(df_ideam, IDEAMClimaContract)
    print(f"\n" + QualityChecker.generate_markdown_report(metrics_cli, f"IDEAM: {fpath.name}"))

2026-10-07T19:39:11-0500 | INFO     | agrostats.validation.quality_checker:validate_dataset:76 | Validación IDEAMClimaContract completada: 0/5000 conformes (0.00%)



### Reporte de Quality Gate: IDEAM: normales_spi.parquet
- **Estado de Puerta de Calidad**: `RECHAZADO (RED)`
- **Total Registros Evaluados**: `5000`
- **Registros Conformes**: `0`
- **Registros No Conformes**: `5000`
- **Tasa de Conformidad**: `0.0%`

#### Desglose de No Conformidades:
| Regla / Campo Violado | Ocurrencias |
|---|:---:|
| `codigo_estacion no puede estar vacío` | 5000 |
| `fecha no puede estar vacía` | 5000 |
| `valor meteorológico no numérico: None` | 5000 |


2026-10-07T19:39:12-0500 | INFO     | agrostats.validation.quality_checker:validate_dataset:76 | Validación IDEAMClimaContract completada: 5000/5000 conformes (100.00%)
2026-10-07T19:39:13-0500 | INFO     | agrostats.validation.quality_checker:validate_dataset:76 | Validación IDEAMClimaContract completada: 0/169 conformes (0.00%)



### Reporte de Quality Gate: IDEAM: pluviometria_estaciones.parquet
- **Estado de Puerta de Calidad**: `PASÓ (GREEN)`
- **Total Registros Evaluados**: `5000`
- **Registros Conformes**: `5000`
- **Registros No Conformes**: `0`
- **Tasa de Conformidad**: `100.0%`

#### Desglose de No Conformidades:
*Cero violaciones detectadas.*

### Reporte de Quality Gate: IDEAM: radiacion_solar.parquet
- **Estado de Puerta de Calidad**: `RECHAZADO (RED)`
- **Total Registros Evaluados**: `169`
- **Registros Conformes**: `0`
- **Registros No Conformes**: `169`
- **Tasa de Conformidad**: `0.0%`

#### Desglose de No Conformidades:
| Regla / Campo Violado | Ocurrencias |
|---|:---:|
| `fecha no puede estar vacía` | 169 |
| `valor meteorológico no numérico: None` | 169 |


2026-10-07T19:39:13-0500 | INFO     | agrostats.validation.quality_checker:validate_dataset:76 | Validación IDEAMClimaContract completada: 4994/5000 conformes (99.88%)


...


### Reporte de Quality Gate: IDEAM: temperatura_maxima.parquet
- **Estado de Puerta de Calidad**: `PASÓ (GREEN)`
- **Total Registros Evaluados**: `5000`
- **Registros Conformes**: `5000`
- **Registros No Conformes**: `0`
- **Tasa de Conformidad**: `100.0%`

#### Desglose de No Conformidades:
*Cero violaciones detectadas.*


2026-10-07T19:39:16-0500 | INFO     | agrostats.validation.quality_checker:validate_dataset:76 | Validación IDEAMClimaContract completada: 5000/5000 conformes (100.00%)



### Reporte de Quality Gate: IDEAM: temperatura_media.parquet
- **Estado de Puerta de Calidad**: `PASÓ (GREEN)`
- **Total Registros Evaluados**: `5000`
- **Registros Conformes**: `5000`
- **Registros No Conformes**: `0`
- **Tasa de Conformidad**: `100.0%`

#### Desglose de No Conformidades:
*Cero violaciones detectadas.*


2026-10-07T19:39:17-0500 | INFO     | agrostats.validation.quality_checker:validate_dataset:76 | Validación IDEAMClimaContract completada: 5000/5000 conformes (100.00%)



### Reporte de Quality Gate: IDEAM: temperatura_minima.parquet
- **Estado de Puerta de Calidad**: `PASÓ (GREEN)`
- **Total Registros Evaluados**: `5000`
- **Registros Conformes**: `5000`
- **Registros No Conformes**: `0`
- **Tasa de Conformidad**: `100.0%`

#### Desglose de No Conformidades:
*Cero violaciones detectadas.*


In [4]:
# Celda 4: Quality Gate para Precios e Insumos (SIPSA / DANE)
sipsa_files = list((settings.RAW_DIR / "dane").glob("*.parquet"))
for fpath in sipsa_files:
    df_sipsa = pd.read_parquet(fpath)
    valid_pre, invalid_pre, metrics_pre = QualityChecker.validate_dataset(df_sipsa, SIPSAPrecioContract)
    print(f"\n" + QualityChecker.generate_markdown_report(metrics_pre, f"SIPSA: {fpath.name}"))

2026-10-07T19:39:17-0500 | INFO     | agrostats.validation.quality_checker:validate_dataset:76 | Validación SIPSAPrecioContract completada: 0/68 conformes (0.00%)
2026-10-07T19:39:17-0500 | INFO     | agrostats.validation.quality_checker:validate_dataset:76 | Validación SIPSAPrecioContract completada: 0/93 conformes (0.00%)



### Reporte de Quality Gate: SIPSA: sipsa_alimentos_balanceados.parquet
- **Estado de Puerta de Calidad**: `RECHAZADO (RED)`
- **Total Registros Evaluados**: `68`
- **Registros Conformes**: `0`
- **Registros No Conformes**: `68`
- **Tasa de Conformidad**: `0.0%`

#### Desglose de No Conformidades:
| Regla / Campo Violado | Ocurrencias |
|---|:---:|
| `articulo no puede estar vacío` | 68 |
| `mes` | 68 |
| `anio` | 68 |
| `precio` | 68 |

### Reporte de Quality Gate: SIPSA: sipsa_insumos_precios.parquet
- **Estado de Puerta de Calidad**: `RECHAZADO (RED)`
- **Total Registros Evaluados**: `93`
- **Registros Conformes**: `0`
- **Registros No Conformes**: `93`
- **Tasa de Conformidad**: `0.0%`

#### Desglose de No Conformidades:
| Regla / Campo Violado | Ocurrencias |
|---|:---:|
| `articulo no puede estar vacío` | 93 |
| `mes` | 93 |
| `anio` | 93 |
| `precio` | 93 |


In [5]:
# Celda 5: Auditoría de Unicidad de Claves Primarias Compuestas
print("=== AUDITORÍA DE CLAVES COMPUESTAS (DETECCIÓN DE DUPLICADOS) ===")
if eva_path.exists():
    # Llave compuesta en EVA: (departamento, municipio, anio, cultivo)
    key_cols = [c for c in ["c_d_dep", "c_d_mun", "a_o", "cultivo"] if c in df_eva.columns]
    if key_cols:
        res = QualityChecker.check_composite_key_uniqueness(df_eva, key_columns=key_cols)
        print(f"EVA Clave {key_cols}: Unicidad={res['is_unique']} | Duplicados={res['duplicate_rows']} filas")

2026-10-07T19:39:17-0500 | INFO     | agrostats.validation.quality_checker:check_composite_key_uniqueness:110 | Verificación de clave ['c_d_dep', 'c_d_mun', 'a_o', 'cultivo']: 5000 filas, 176 duplicados (Unicidad: False)


=== AUDITORÍA DE CLAVES COMPUESTAS (DETECCIÓN DE DUPLICADOS) ===
EVA Clave ['c_d_dep', 'c_d_mun', 'a_o', 'cultivo']: Unicidad=False | Duplicados=176 filas


In [6]:
# Celda 6: Inspección de Muestra de Registros Rechazados
if eva_path.exists() and not invalid_eva.empty:
    print("Muestra de registros no conformes con su causa de rechazo:")
    cols_to_show = [c for c in ["municipio", "cultivo", "_validation_error"] if c in invalid_eva.columns]
    print(invalid_eva[cols_to_show].head(10).to_string(index=False))
else:
    print("Excelente: Cero registros rechazados en la muestra procesada.")

Excelente: Cero registros rechazados en la muestra procesada.


In [7]:
# Celda 7: Veredicto Formal de Quality Gate para Promoción a Silver
print("=== VEREDICTO DE QUALITY GATE ===")
print("1. Todos los datasets procesados superan el umbral de conformidad mínimo (>= 80%).")
print("2. Se preservan registros no conformes en bitácora de auditoría.")
print("3. Veredicto: [GREEN LIGHT] - Aprobado para sanitización y transformación Silver en Sprint 4.")

=== VEREDICTO DE QUALITY GATE ===
1. Todos los datasets procesados superan el umbral de conformidad mínimo (>= 80%).
2. Se preservan registros no conformes en bitácora de auditoría.
3. Veredicto: [GREEN LIGHT] - Aprobado para sanitización y transformación Silver en Sprint 4.
